#### RAG Pipeline - Data Ingestion to Vector DB Pipeline 

In [1]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\aabha\AppData\Local\Temp\ipykernel_11268\3728471183.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\aabha\OneDrive\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#### Read all the pdf files 
def process_all_pdfs(pdf_directory):
    """ Process all the pdf files in a directory """
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all the pdf files recursively 
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata 
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  loaded {len(documents)} pages")

        

        except Exception as e:
            print(f"   Error:{e}")

    print(f"\nTotal documents loaded : {len(all_documents)}")
    return all_documents

# Process all the pdfs in the data directory
all_pdf_documents = process_all_pdfs("../data")


found 5 PDF files to process

Processing: fepr101.pdf
  loaded 38 pages

Processing: fepr102.pdf
  loaded 36 pages

Processing: fepr103.pdf
  loaded 28 pages

Processing: fepr104.pdf
  loaded 28 pages

Processing: fepr1ps.pdf
  loaded 16 pages

Total documents loaded : 146


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2024-12-09T15:47:55+05:30', 'moddate': '2026-03-27T11:31:30+05:30', 'trapped': '/False', 'source': '..\\data\\pdf\\fepr101.pdf', 'total_pages': 38, 'page': 0, 'page_label': '1', 'source_file': 'fepr101.pdf', 'file_type': 'pdf'}, page_content='A Bottle of Dew\nLet us do these activities before we read.\nI Circle the picture that matches with each word. Check your answers by \nsharing them with your classmates and teacher.\n1. worried\n2. plantation\n3. sage\n4. surprise\nII Answer these questions and \ndiscuss them with your classmates and teacher.\n1. Think of a time when you worked hard. What did you do then?\n2. How did it help you?\n3. How did it make you feel?\nFabLes and FoLk TaLes\nUnit 1\nUnit 1.indd   1 09-Dec-24   3:47:58 PM\nReprint 2026-27'),
 Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate

In [4]:
### Text spilitting get into chunks

def split_documents(documents, chunk_size=1000,chunk_overlap=200):
    """ Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk 
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)

split 146 documents into 230 chunks

Example chunk:
Content: A Bottle of Dew
Let us do these activities before we read.
I Circle the picture that matches with each word. Check your answers by 
sharing them with your classmates and teacher.
1. worried
2. plantat...
Metadata: {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2024-12-09T15:47:55+05:30', 'moddate': '2026-03-27T11:31:30+05:30', 'trapped': '/False', 'source': '..\\data\\pdf\\fepr101.pdf', 'total_pages': 38, 'page': 0, 'page_label': '1', 'source_file': 'fepr101.pdf', 'file_type': 'pdf'}


#### Emebedding and vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

        """ Initialize the embedding manager
        Args:
            model_name:HuggingFace model name for sentence embeddings
        """
        self.model_name=model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentenceTranformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
        texts: List of the text strings to embed

        Returns:
        numpy array of embeddings with shape (len(texts), embediing_dim)
        """

        if not self.model:
            raise ValueError("model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape : {embeddings.shape})")
        return embeddings

    # def get_embedding_dimension(self) -> int:
    #     """ GEt the embedding dimension of the model """
    #     if not self.model:
    #         raise ValueError("Model not loaded")
    #     return self.model.get_sentence_embedding_dimension()

### Initialize the embeddings manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2973.52it/s]


model loaded successfully. Embedding dimension: 384


C:\Users\aabha\AppData\Local\Temp\ipykernel_11268\1050751685.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


#### VectorStore

In [9]:
class VectorStore:
    """ Manages document embeddings in a Chromadb vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory:str = "../data/vector_store"):
        """
        Initializes the vector store
        
        Args:
            Collection_name:Name of the ChromadDB collection 
            persist_directory: Directory to persist the vector store
            """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None 
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Intialize ChromaDB client and collection """
        try:
            # Create persistent ChromaDB client 
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection 
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embedding for RAG"}

            )
            print(f"Vector store intialized. Collection:{self.collection_name}")
            print(f"Existing documents inj collection : {self.collection.count()}")

        except Exception as e:
            print(f"Error initalizing vector stor: {e}")
            raise

    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embedding for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list =[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i 
            metadata['content_length'] = len(doc.page_content)

            # Add metadata

            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

           

        # Add to collection 
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore=VectorStore()
vectorstore



Vector store intialized. Collection:pdf_documents
Existing documents inj collection : 7364


In [10]:
# import os
# from dotenv import load_dotenv
# load_dotenv()

# print(os.getenv("GROQ_API_KEY"))

In [11]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2024-12-09T15:47:55+05:30', 'moddate': '2026-03-27T11:31:30+05:30', 'trapped': '/False', 'source': '..\\data\\pdf\\fepr101.pdf', 'total_pages': 38, 'page': 0, 'page_label': '1', 'source_file': 'fepr101.pdf', 'file_type': 'pdf'}, page_content='A Bottle of Dew\nLet us do these activities before we read.\nI Circle the picture that matches with each word. Check your answers by \nsharing them with your classmates and teacher.\n1. worried\n2. plantation\n3. sage\n4. surprise\nII Answer these questions and \ndiscuss them with your classmates and teacher.\n1. Think of a time when you worked hard. What did you do then?\n2. How did it help you?\n3. How did it make you feel?\nFabLes and FoLk TaLes\nUnit 1\nUnit 1.indd   1 09-Dec-24   3:47:58 PM\nReprint 2026-27'),
 Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate

In [12]:
### Convert the text to Embeddings 
texts = [doc.page_content for doc in chunks]

## Generate the Embeddings 

embeddings=embedding_manager.generate_embeddings(texts)
print(len(chunks))
print(embeddings.shape)

## Store in the vecotre database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 230 texts...


Batches: 100%|██████████| 8/8 [00:09<00:00,  1.18s/it]


Generated embeddings with shape : (230, 384))
230
(230, 384)
Adding 230 documents to vector store...
Successfully added 230 documents to vector store
Total documents in collection: 7594


In [13]:
class RAGRetriever:
    """ Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Intialize the retriever 

        Args:
            vector_store: Vector store containing document embeddings 
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold:float =0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query:The search query
            top_k:
            Number of top results to return 
            score_threshold: Minimum similarity score threshold

            Returns:
            List of dictionaries containing retrieved documents and metadata 
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # GEnerating query embedding 
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store 
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # process results
            retrieved_docs =[]

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents,metadatas, distances)):
                    # convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents foound")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return[]

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [14]:
rag_retriever

In [15]:
rag_retriever.retrieve("What is attention")

Retrieving documents for query: 'What is attention'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 78.96it/s]

Generated embeddings with shape : (1, 384))
Retrieved 5 documents (after filtering)


[{'id': 'doc_280b5e1c_452',
  'content': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'subject': '',
   'creator': 'LaTeX with hyperref',
   'source': '..\\data\\pdf\\03.Attention Is All You Need.pdf',
   'content_length': 216,
   'page_label': '3',
   'producer': 'pdfTeX-1.40.25',
   'author': '',
   'title': '',
   'trapped': '/False',
   'page': 2,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'moddate': '2024-04-10T21:11:43+00:00',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'doc_index': 452,
   'file_type': 'pdf',
   'total_pages': 15,
   'source_file': '03.Attention Is All You Need.pdf',
   'keywords': ''},
  'similarity_score': 0.40325474739074707,
  'distance': 0.5967452526092529,
  'rank': 1},
 {'id': 'doc_3435e3

#### Enhanced RAG Pipeline Features 

In [20]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv

# 1. Force load the API key
load_dotenv(override=True)
raw_key = os.getenv("GROQ_API_KEY")
api_key = raw_key.strip() if raw_key else ""

# 2. Initialize with the ACTIVE GPT-OSS model!
llm = ChatGroq(
    groq_api_key=api_key,
    model_name="openai/gpt-oss-20b",  # <-- The brand new replacement model
    temperature=0.1,
    max_tokens=1024
)

# 3. Your Advanced RAG Function
def rag_advance(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.','sources':[],'confidence':0.0,'context':''}

    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page': doc['metadata'].get('page','unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    
    confidence = max([doc['similarity_score'] for doc in results])

    prompt_template = """Use the following context to answer the question concisely if question demands an explanation then explain it. \n{context}\n\nQuestion: {query}\n\nAnswer:""" 
    
    response = llm.invoke([prompt_template.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# 4. Execute the query
print("--- Testing Pipeline with GPT-OSS 20B ---")
result = rag_advance(
    query="the RAven AnD the fox explain in 300 words properly in childs language", 
    retriever=rag_retriever, 
    llm=llm, 
    top_k=3, 
    min_score=0.1, 
    return_context=True
)

print("\nAnswer:", result['answer']) 
print("Sources:", result['sources'])

--- Testing Pipeline with GPT-OSS 20B ---
Retrieving documents for query: 'the RAven AnD the fox explain in 300 words properly in childs language'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.42it/s]

Generated embeddings with shape : (1, 384))
Retrieved 3 documents (after filtering)



Answer: **The Story of the Raven and the Fox – A Simple Explanation for Kids**

Once upon a time, in a green forest, lived a proud raven named Mr. Raven. He loved to show off his shiny black feathers and brag about how smart he was. One sunny day, a sly fox named Reynard (the fox’s real name) was watching the raven from a tree.

Reynard had a tasty piece of food in his mouth. He wanted to trick the raven into giving it to him. “Hey, Mr. Raven,” he called, “you look so clever! Can you sing for me?” The raven, proud and eager to prove himself, opened his beak to sing. But instead of singing, he let the food fall out of his beak and onto the ground. The fox laughed loudly, “Ha‑ha! You’re so foolish!” The raven felt very embarrassed.

The fox then said, “Don’t let sweet words make you feel too proud. Pride can be unwise.” The raven realized that he had let his pride get in the way of his judgment. He had forgotten that he should be careful and not let others trick him.

From that day on, 

In [16]:
# # ---- Enhanced RAG Pipeline Features ----
# def rag_advance(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
#     """
#     RAG pipeline with extra features :
#     - Returns answer, sources, confidence score, and optionally full context.
#     """
#     results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
#     if not results:
#         return {'answer': 'No relevant context found.','sources':[],'confidence':0.0,'context':''}

#     # prepare context and sources 
#     context = "\n\n".join([doc['content'] for doc in results])
#     sources = [{
#         'source': doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
#             'page': doc['metadata'].get('page','unknown'),
#             'score': doc['similarity_score'],
#             'preview': doc['content'][:300] + '...'
#         } for doc in results
#         ]
#     confidence = max([doc['similarity_score'] for doc in results])

#     # Generator Prompt 
#     prompt = f"""use the following context to answer the question concisely if question demands an explanation then explain it. \n{context}\n\nQuestion: {query}\n\nAnswer:""" 
#     response = llm.invoke([prompt.format(context=context, query=query)])

#     output = {
#         'answer': response.content,
#         'sources': sources,
#         'confidence': confidence
#     }
#     if return_context:
#         output['context'] = context
#     return output

# # Example usage :
# result = rag_advance("what is the attention mechanism ?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
# print("Answer:", result['answer']) 
# print("Sources:", result['sources'])
# print("confidence:", result['confidence'])
# print("Context preview:", result['context'][:300])

In [17]:
# print(llm.invoke("Hello").content)

In [18]:
# import os
# from langchain_groq import ChatGroq
# from dotenv import load_dotenv

# # 1. Force load the new key
# load_dotenv(override=True)
# api_key = os.getenv("GROQ_API_KEY")

# print(f"Verifying Key: {api_key[:8]}...") # Should print gsk_Yht2...

# # 2. CREATE A BRAND NEW LLM INSTANCE HERE
# fresh_llm = ChatGroq(
#     groq_api_key=api_key,
#     model_name="gemma2-9b-it",
#     temperature=0.1,
#     max_tokens=1024
# )

# # 3. Your Advanced RAG Function (with the string formatting fixed)
# def rag_advance(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
#     results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
#     if not results:
#         return {'answer': 'No relevant context found.','sources':[],'confidence':0.0,'context':''}

#     context = "\n\n".join([doc['content'] for doc in results])
#     sources = [{
#         'source': doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
#         'page': doc['metadata'].get('page','unknown'),
#         'score': doc['similarity_score'],
#         'preview': doc['content'][:300] + '...'
#     } for doc in results]
    
#     confidence = max([doc['similarity_score'] for doc in results])

#     # FIX: Removed the 'f' string prefix so .format() works safely without crashing on {} in PDF text
#     prompt_template = """Use the following context to answer the question concisely if question demands an explanation then explain it. \n{context}\n\nQuestion: {query}\n\nAnswer:""" 
    
#     response = llm.invoke([prompt_template.format(context=context, query=query)])

#     output = {
#         'answer': response.content,
#         'sources': sources,
#         'confidence': confidence
#     }
#     if return_context:
#         output['context'] = context
#     return output

# # 4. Test it with the FRESH LLM
# print("\n--- Testing Pipeline ---")
# result = rag_advance(
#     query="what is the attention mechanism ?", 
#     retriever=rag_retriever, 
#     llm=fresh_llm, # Passing the newly created LLM
#     top_k=3, 
#     min_score=0.1, 
#     return_context=True
# )

# print("\nAnswer:", result['answer']) 
# print("Sources:", result['sources'])

In [19]:
# import os
# import requests
# from dotenv import load_dotenv

# # 1. Force reload
# load_dotenv(override=True)

# # 2. Get the key and STRIP any accidental spaces or hidden newline characters
# raw_key = os.getenv("GROQ_API_KEY")
# api_key = raw_key.strip() if raw_key else ""

# print(f"Testing Key: {api_key[:8]}... (Length: {len(api_key)})")

# # 3. Direct API call to Groq bypassing LangChain
# url = "https://api.groq.com/openai/v1/chat/completions"
# headers = {
#     "Authorization": f"Bearer {api_key}",
#     "Content-Type": "application/json"
# }
# data = {
#     "model": "gemma2-9b-it",
#     "messages": [{"role": "user", "content": "Just say hi!"}]
# }

# print("Sending request to Groq...")
# response = requests.post(url, headers=headers, json=data)

# print(f"\nStatus Code: {response.status_code}")
# if response.status_code == 200:
#     print("Success! Response:", response.json()['choices'][0]['message']['content'])
# else:
#     print("FAILED. Error details:", response.text)

In [20]:
# from dotenv import load_dotenv
# import os

# load_dotenv(override=True)

# print("LLM_API_KEY exists:", os.getenv("LLM_API_KEY") is not None)
# print("GROQ_API_KEY exists:", os.getenv("GROQ_API_KEY") is not None)

# key = os.getenv("GROQ_API_KEY") or os.getenv("LLM_API_KEY")

# print("Key selected:", key is not None)

# if key:
#     print("Prefix:", key[:4])
#     print("Length:", len(key))

In [21]:
# import os
# from langchain_groq import ChatGroq
# from dotenv import load_dotenv

# # 1. Load the key (using .strip() just in case there were hidden spaces!)
# load_dotenv(override=True)
# raw_key = os.getenv("GROQ_API_KEY")
# api_key = raw_key.strip() if raw_key else ""

# # 2. Initialize with an ACTIVE model
# llm = ChatGroq(
#     groq_api_key=api_key,
#     model_name="llama3-8b-8192", # <--- CHANGED FROM gemma2-9b-it
#     temperature=0.1,
#     max_tokens=1024
# )

# # 3. Test your simple pipeline again
# def rag_simple(query, retriever, llm, top_k=3):
#     results = retriever.retrieve(query, top_k=top_k)
#     context = "\n\n".join([doc['content'] for doc in results]) if results else ""
#     if not context:
#         return "No relevant context found to answer the question."
    
#     prompt = f"""Use the following context to answer the question concisely.
#         Context:
#         {context}

#         Question: {query}

#         Answer:"""
    
#     response = llm.invoke([prompt.format(context=context, query=query)])
#     return response.content

# print("Testing simple RAG...")
# answer = rag_simple("What is attention mechanism?", rag_retriever, llm)
# print(answer)